# Lab 5 · CI/CD: Git folders, bundles and the Databricks CLI

Clone a repository into a Git folder and work on a branch, write a bundle with dev and prod targets and a catalog variable, then validate, deploy and run it with the Databricks CLI.

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Implementing CI/CD (10% of the exam) · **Time:** about 35 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## Part 1 · A Git folder (UI, no code)

1. Open **Workspace**, then your home folder, then **Create → Git folder**.
2. Paste `https://github.com/datapreplab/lakehouse-lab` as the Git repository URL and click **Create Git folder**. A public repository clones without credentials.
3. Click the **branch name** button at the top of the folder. In the Git dialog, create a branch called `my-first-branch`.
4. Open `README.md` in the folder, change a line, and look at the Git dialog again: your change is listed, ready to **commit and push**. Pushing needs a Git credential and a repository you can write to, so stop here, or fork the repository first.

The Git dialog is where you pull, create branches, commit, push and resolve merge conflicts. Pull requests are opened in your Git provider.

## Part 2 · Write a small bundle

A bundle describes resources such as jobs and pipelines in `databricks.yml`. **Targets** (dev, prod) override settings, and **variables** carry the values that differ, so one codebase deploys everywhere. This cell writes the bundle files into your workspace so you can read them; Part 3 deploys them from your laptop.

In [ ]:
me = spark.sql("SELECT current_user()").first()[0]
root = f"/Workspace/Users/{me}/dea_lab_bundle"

files = {
"databricks.yml": """bundle:
  name: dea_lab_bundle

variables:
  catalog:
    description: Catalog the job writes to
    default: workspace

include:
  - resources/*.yml

targets:
  dev:
    mode: development        # prefixes resource names with [dev <you>] and pauses schedules
    default: true
  prod:
    mode: production
    workspace:              # production mode wants one fixed deployment path
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}
    variables:
      catalog: workspace      # in a real company: prod_catalog
""",
"resources/daily_orders.job.yml": """resources:
  jobs:
    daily_orders:
      name: dea_lab_bundle_daily_orders
      parameters:
        - name: catalog
          default: ${var.catalog}
      tasks:
        - task_key: build
          notebook_task:
            notebook_path: ../src/build.py
""",
"src/build.py": """# Databricks notebook source
catalog = dbutils.widgets.get("catalog")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.dea_lab")
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.dea_lab.bundle_orders AS SELECT id AS order_id FROM range(3)")
print("built", catalog + ".dea_lab.bundle_orders")
""",
}
import os
for name, text in files.items():
    path = f"{root}/{name}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)
print("Bundle written to", root)

Open the `dea_lab_bundle` folder in your workspace and read `databricks.yml`. Notice there is **no catalog name hard-coded** in the job or the notebook; `${var.catalog}` is filled in per target.

## Part 3 · Validate, deploy and run (on your laptop)

Install the Databricks CLI on your computer (see the CLI docs for your operating system), then sign in to your Free Edition workspace. Copy the workspace URL from your browser's address bar.

```bash
databricks auth login --host https://<your-workspace-url> --profile DEFAULT
```

Download the `dea_lab_bundle` folder (or copy the three files above into a folder with the same layout), then run from inside it:

```bash
databricks bundle validate                 # checks the YAML and shows the resolved settings
databricks bundle deploy -t dev            # creates or updates the job; runs nothing
databricks bundle run -t dev daily_orders  # starts the job and waits for it
databricks bundle deploy -t prod --var="catalog=workspace"
```

In **Jobs & Pipelines** you will see `[dev <your name>] dea_lab_bundle_daily_orders` from the dev target and `dea_lab_bundle_daily_orders` from prod: the same code, two environments.

In a CI pipeline, the same `validate`, `deploy` and `run` commands run with a **service principal**, and secrets come from the CI system's environment, never from the YAML.

## Part 4 · Check the result

In [ ]:
%sql
SELECT * FROM workspace.dea_lab.bundle_orders ORDER BY order_id;

## Part 5 · Clean up

```bash
databricks bundle destroy -t dev
databricks bundle destroy -t prod
```

In [ ]:
import shutil
shutil.rmtree(root, ignore_errors=True)
print("Bundle files removed from the workspace")

## Expected results

```text
Part 1: the Git folder shows the repository files, and the branch button reads my-first-branch.
Part 2: dea_lab_bundle has databricks.yml, resources/daily_orders.job.yml and src/build.py.
Part 3: validate prints the bundle name, target dev and the workspace paths. deploy ends with
        "Deployment complete!". run waits for the job and reports the build task as succeeded.
Part 4: bundle_orders has 3 rows: 0, 1, 2.
```

## Exam links

- One codebase with target overrides is the intended answer; copying code per environment is the anti-pattern.
- `bundle deploy` creates or updates resources but runs nothing; `bundle run` starts a job or pipeline.
- Development mode prefixes resource names and pauses schedules; production mode is for the shared, scheduled deployment.
- Pass environment-specific values, such as catalogs, through bundle variables.